# Fetch Ingested Videos
Fetches recently (last 24 hours) ingested videos and prints 10 random videos with views.

In [ ]:
# Install pymongo if not already present
!pip install pymongo dnspython

import pymongo
from pymongo import MongoClient
from google.colab import userdata

try:
    # Retrieve the URI from Colab Secrets
    uri = userdata.get('MONGODB_URI')
    client = MongoClient(uri)

    # Send a ping to confirm a successful connection
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(f"An error occurred: {e}")
    print("\nMake sure you have added 'MONGODB_URI' to your Colab Secrets (the key icon on the left).")

In [ ]:
from bson.objectid import ObjectId
import datetime
import random

# Predikto database and collection mapping per documentation (assumed generic Colab use case since this is an analytics notebook similar to Graphiko)
db = client['predikto']
videos_col = db['publicvideos']

# Fetch videos ingested in the last 24 hours using timezone-aware UTC datetime
twenty_four_hours_ago = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=1)
oid = ObjectId.from_datetime(twenty_four_hours_ago)
query = {'_id': {'$gte': oid}}

# Use aggregate to randomly sample 10 videos
pipeline = [
    {'$match': query},
    {'$sample': {'size': 10}}
]
recent_videos = list(videos_col.aggregate(pipeline))

if not recent_videos:
    print('No videos found in the last 24 hours.')
else:
    print(f'Found {len(recent_videos)} videos in the random sample. Printing titles and view counts:')
    for video in recent_videos:
        title = video.get('title', 'Unknown Title')
        views = video.get('viewCount', 0)
        print(f'- {title} | Views: {views}')


In [ ]:
# Install InfluxDB client and pandas
!pip install influxdb-client pandas

### Create 10k Training Dataset using InfluxDB
Queries `channel_metrics` to find the first snapshot (T=0) and the snapshot approximately 24 hours later (T=24) for each video.

In [ ]:
import pandas as pd
from influxdb_client import InfluxDBClient
from google.colab import userdata

try:
    # Retrieve InfluxDB connection details from Colab Secrets
    url = userdata.get('INFLUXDB_URL')
    token = userdata.get('INFLUXDB_TOKEN')
    org = userdata.get('INFLUXDB_ORG')
    bucket = userdata.get('INFLUXDB_BUCKET')

    client = InfluxDBClient(url=url, token=token, org=org)
    query_api = client.query_api()
    print("Successfully connected to InfluxDB!")
except Exception as e:
    print(f"An error occurred during InfluxDB connection: {e}")
    print("\nMake sure you have added INFLUXDB_URL, INFLUXDB_TOKEN, INFLUXDB_ORG, and INFLUXDB_BUCKET to your Colab Secrets.")

# Flux query to retrieve videoId and views from channel_metrics
# We pivot the data to get views as a column for each snapshot
flux_query = f'''
    from(bucket: "{bucket}")
        |> range(start: 0) // Query all available data
        |> filter(fn: (r) => r["_measurement"] == "channel_metrics")
        |> filter(fn: (r) => r["_field"] == "views" or r["_field"] == "videoId")
        |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
        |> keep(columns: ["_time", "videoId", "views"])
'''

try:
    print("Querying InfluxDB... This might take a while.")
    # Query and load directly into a pandas DataFrame
    df = query_api.query_data_frame(flux_query)
    
    # Handle cases where multiple tables are returned (df is a list)
    if isinstance(df, list):
        df = pd.concat(df, ignore_index=True)

    # Convert _time to datetime
    df['_time'] = pd.to_datetime(df['_time'])
    
    # Group by videoId to find T=0 and T=24h
    grouped = df.groupby('videoId')
    
    training_data = []
    
    for video_id, group in grouped:
        # Sort by time
        group = group.sort_values(by='_time')
        
        if len(group) < 2:
            continue # Need at least two snapshots
            
        t0_row = group.iloc[0]
        t0_time = t0_row['_time']
        
        # Find the snapshot closest to 24 hours later
        target_time = t0_time + pd.Timedelta(hours=24)
        
        # Calculate absolute time difference from 24h
        group['time_diff'] = abs(group['_time'] - target_time)
        
        # Filter out the T0 snapshot itself before finding the closest
        future_snapshots = group[group['_time'] > t0_time]
        
        if future_snapshots.empty:
             continue
        
        t24_row = future_snapshots.loc[future_snapshots['time_diff'].idxmin()]
        
        # Check if the closest snapshot is reasonably close to 24 hours (e.g., within 4 hours)
        if t24_row['time_diff'] <= pd.Timedelta(hours=4):
             training_data.append({
                 'videoId': video_id,
                 'views_t0': t0_row['views'],
                 'views_t24': t24_row['views'],
                 'time_t0': t0_row['_time'],
                 'time_t24': t24_row['_time']
             })
             
        # Stop if we hit 10k
        if len(training_data) >= 10000:
             break
             
    training_dataset = pd.DataFrame(training_data)
    
    print(f"\nCreated training dataset with {len(training_dataset)} rows.")
    print("\n--- Sample Dataset ---")
    print(training_dataset.head())
    
    # Debugging input for a single video
    if not training_dataset.empty:
        sample_video_id = training_dataset.iloc[0]['videoId']
        print(f"\n--- Debugging Input for video {sample_video_id} ---")
        sample_group = df[df['videoId'] == sample_video_id].sort_values(by='_time')
        print(sample_group[['_time', 'views']])
        
except Exception as e:
    print(f"An error occurred during data processing: {e}")
